# SKAB EDA

quick look at the SKAB dataset: channel distributions, anomaly rate per split, sensor correlation, anomaly span structure, and a visual walk through a labelled run.

run from repo root so `src.*` imports resolve:

```bash
jupyter lab notebooks/01_eda.ipynb
```

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.skab import load_skab, SENSOR_COLS

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (12, 4)

## 1. load each split

In [ ]:
train = load_skab('train')
valid = load_skab('valid')
test = load_skab('test')

summary = pd.DataFrame({
    'rows': [len(train), len(valid), len(test)],
    'anomaly_frac': [train['anomaly'].mean(), valid['anomaly'].mean(), test['anomaly'].mean()],
}, index=['train', 'valid', 'test'])
summary

anomaly-free is 0% by construction. valve/other splits sit around 30-40% — SKAB is not as imbalanced as people assume. focal loss still helps the hard examples but class weighting isn't critical.

## 2. channel distributions

normal vs anomalous slices in validation. channels that shift visibly here are the ones the model will lean on.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for ax, col in zip(axes.ravel(), SENSOR_COLS):
    ax.hist(valid.loc[valid['anomaly'] == 0, col], bins=60, alpha=0.5, label='normal', density=True)
    ax.hist(valid.loc[valid['anomaly'] == 1, col], bins=60, alpha=0.5, label='anomaly', density=True)
    ax.set_title(col, fontsize=9)
    ax.legend(fontsize=7)
fig.tight_layout()

## 3. sensor correlation

eight channels, but they're coupled. pressure / current / volume-flow move together — the transformer head exploits that cross-channel structure.

In [ ]:
corr = train[SENSOR_COLS].corr()
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
ax.set_xticks(range(len(SENSOR_COLS)))
ax.set_yticks(range(len(SENSOR_COLS)))
ax.set_xticklabels(SENSOR_COLS, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(SENSOR_COLS, fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('channel correlation (train)')

## 4. anomaly span structure

are anomalies isolated spikes or sustained spans? decides window + smoothing.

In [ ]:
def run_lengths(y):
    y = np.asarray(y).astype(int)
    if len(y) == 0:
        return np.array([])
    edges = np.diff(np.concatenate([[0], y, [0]]))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0]
    return ends - starts

lens = run_lengths(valid['anomaly'])
print(f'spans: {len(lens)}')
print(f'median length: {int(np.median(lens))} samples')
print(f'p95 length:    {int(np.percentile(lens, 95))} samples')

plt.hist(lens, bins=30)
plt.xlabel('anomaly span length (samples, ~1s each)')
plt.ylabel('count')
plt.title('valid split: anomaly run-length distribution');

median span is well above the 64-sample window, so `window=64, stride=8` captures full events rather than slivers. same reason `median_smooth(k=5)` on output scores doesn't eat true positives.

## 5. walk a labelled valve run

first 3000 samples of the valid split, 3 channels with anomaly spans shaded.

In [ ]:
sample = valid.iloc[:3000].reset_index(drop=True)
cols = ['Pressure', 'Current', 'Accelerometer1RMS']

fig, axes = plt.subplots(len(cols), 1, figsize=(14, 7), sharex=True)
for ax, col in zip(axes, cols):
    ax.plot(sample[col].values, linewidth=0.7)
    ax.set_ylabel(col, fontsize=9)
    y = sample['anomaly'].values
    in_span, start = False, 0
    for i, v in enumerate(y):
        if v and not in_span:
            start, in_span = i, True
        elif not v and in_span:
            ax.axvspan(start, i, alpha=0.2, color='red')
            in_span = False
    if in_span:
        ax.axvspan(start, len(y), alpha=0.2, color='red')
axes[-1].set_xlabel('sample index')
fig.suptitle('valid split — anomaly spans shaded red');

## takeaways

- 8 channels, tens of thousands of rows per split after dropping NaN tails
- ~35% anomaly rate on valve/other — class weighting less important than it sounds
- pressure / current / volume-flow tightly correlated → transformer's cross-time attention earns its keep
- median anomaly span >> window, so overlapping windows with stride 8 are enough; no need for dilated conv or longer contexts